# Kalman R7 Point-in-Time Information Readiness

Research-only. This notebook does not fit a model and does not change LIVE/production.


In [ ]:
from google.colab import userdata
import os, subprocess, shutil, pathlib

REPO='/content/Kalman_R7'
BRANCH='research/r7-point-in-time-info-20260922'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/kimtk94/Codex.git',REPO],check=True)
subprocess.run(['pip','-q','install','psycopg[binary]'],check=True)
head=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('HEAD =', head)

dsn = userdata.get('NEON_DATABASE_URL')
if not dsn:
    raise RuntimeError('Add NEON_DATABASE_URL in Colab Secrets')
env=os.environ.copy(); env['NEON_DATABASE_URL']=dsn
script=f'{REPO}/kalman-toss-gateway/research/r7_point_in_time_readiness.py'
tests=f'{REPO}/kalman-toss-gateway/tests/test_r7_point_in_time_readiness.py'
subprocess.run(['python','-m','py_compile',script],check=True)
subprocess.run(['pytest','-q',tests],check=True)
run=subprocess.run(['python',script],env=env,text=True,capture_output=True)
print(run.stdout)
if run.stderr: print(run.stderr)
if run.returncode != 0: raise RuntimeError(f'R7 readiness failed with exit code {run.returncode}')
